In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Executar jobs em um batch

<details>
<summary><b>Versões dos pacotes</b></summary>

O código desta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar estas versões ou versões mais recentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Use o modo batch para enviar múltiplos jobs primitivos simultaneamente. A seguir estão exemplos de como trabalhar com batches.

## Configurar para usar batches
Antes de iniciar um batch, você deve [configurar o Qiskit Runtime](/guides/install-qiskit) e inicializá-lo como um serviço:

In [1]:
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    Batch,
    SamplerV2 as Sampler,
    EstimatorV2 as Estimator,
)


service = QiskitRuntimeService()

## Abrir um batch
Você pode abrir um batch de runtime usando o gerenciador de contexto `with Batch(...)` ou inicializando a classe `Batch`.
Ao iniciar um batch, você deve especificar um QPU passando um objeto `backend`. O batch começa quando seu primeiro job inicia a execução.

**Classe Batch**

In [2]:
backend = service.least_busy(operational=True, simulator=False)
batch = Batch(backend=backend)
estimator = Estimator(mode=batch)
sampler = Sampler(mode=batch)
# Close the batch because no context manager was used.
batch.close()

**Gerenciador de contexto**

O gerenciador de contexto abre e fecha o batch automaticamente.

In [3]:
from qiskit_ibm_runtime import (
    Batch,
    SamplerV2 as Sampler,
    EstimatorV2 as Estimator,
)

backend = service.least_busy(operational=True, simulator=False)
with Batch(backend=backend):
    estimator = Estimator()
    sampler = Sampler()

<span id="specify-batch-length"></span>
## Duração do batch
Você pode definir o tempo máximo de vida (TTL) do batch com o parâmetro `max_time`. Esse valor deve superar o tempo de execução do job mais longo. O temporizador começa quando o batch é iniciado. Quando o valor é atingido, o batch é encerrado. Quaisquer jobs em execução serão finalizados, mas os jobs ainda na fila serão marcados como falhos.

In [4]:
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit import QuantumCircuit, Parameter
from qiskit.transpiler import generate_preset_pass_manager
import numpy as np

# This cell is hidden from users
service = QiskitRuntimeService()
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
transpiled_circuit_sampler = transpiled_circuit
transpiled_circuit_sampler.measure_all()

params = np.random.uniform(size=(2, 3)).T
observables = [
    [
        SparsePauliOp(["XX", "IY"], [0.5, 0.5]).apply_layout(
            transpiled_circuit.layout
        )
    ],
    [SparsePauliOp("XX").apply_layout(transpiled_circuit.layout)],
    [SparsePauliOp("IY").apply_layout(transpiled_circuit.layout)],
]

sampler_pub = (transpiled_circuit_sampler, params)
estimator_pub = (transpiled_circuit_sampler, observables, params)

In [5]:
with Batch(backend=backend) as batch:
    estimator = Estimator()
    sampler = Sampler()
    job1 = estimator.run([estimator_pub])
    job2 = sampler.run([sampler_pub])

# The batch is no longer accepting jobs but the submitted job will run to completion.
result = job1.result()
result2 = job2.result()

<Admonition type="tip">
If you are not using a context manager, manually close the batch.  If you leave the batch open and submit more jobs to it later, it is possible that the maximum TTL will be reached before the subsequent jobs start running; causing them to be canceled. You can close a batch as soon as you are done submitting jobs to it. When a batch is closed with `batch.close()`, it no longer accepts new jobs, but the already submitted jobs will still run until completion and their results can be retrieved.
</Admonition>

In [6]:
batch = Batch(backend=backend)

# If using qiskit-ibm-runtime earlier than 0.24.0, change `mode=` to `batch=`
estimator = Estimator(mode=batch)
sampler = Sampler(mode=batch)
job1 = estimator.run([estimator_pub])
job2 = sampler.run([sampler_pub])
print(f"Result1: {job1.result()}")
print(f"Result2: {job2.result()}")

# Manually close the batch. Running and queued jobs will run to completion.
batch.close()

Result1: PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 2), dtype=float64>), stds=np.ndarray(<shape=(3, 2), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 2), dtype=float64>), shape=(3, 2)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})


Result2: PrimitiveResult([SamplerPubResult(data=DataBin(meas=BitArray(<shape=(3, 2), num_shots=4096, num_bits=2>), meas0=BitArray(<shape=(3, 2), num_shots=4096, num_bits=133>), shape=(3, 2)), metadata={'circuit_metadata': {}})], metadata={'execution': {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2026-01-15 07:47:58', stop='2026-01-15 07:48:05', size=24576>)])}, 'version': 2})


> **Tip:** Se você não estiver usando um gerenciador de contexto, feche o batch manualmente. Se você deixar o batch aberto e enviar mais jobs para ele posteriormente, é possível que o TTL máximo seja atingido antes que os jobs subsequentes comecem a ser executados, fazendo com que sejam cancelados. Você pode fechar um batch assim que terminar de enviar jobs para ele. Quando um batch é fechado com `batch.close()`, ele não aceita mais novos jobs, mas os jobs já enviados continuarão sendo executados até a conclusão e seus resultados poderão ser recuperados.

In [7]:
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    batch,
    SamplerV2 as Sampler,
)

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

with Batch(backend=backend) as batch:
    print(batch.details())

{'id': 'ce8cf08d-b18e-4d56-ab51-eaff0b8190f4', 'backend_name': 'ibm_torino', 'interactive_timeout': 1, 'max_time': 28800, 'active_timeout': 28800, 'state': 'open', 'accepting_jobs': True, 'last_job_started': None, 'last_job_completed': None, 'started_at': None, 'closed_at': None, 'activated_at': None, 'mode': 'batch', 'usage_time': None}


<span id="partition"></span>
## Reconfigure jobs for parallel processing

There are multiple ways you can reconfigure your jobs to take advantage of the parallel processing provided by batching. The following example shows how you can partition a long list of circuits into multiple jobs and run them as a batch to take advantage of the parallel processing.

In [8]:
from qiskit_ibm_runtime import SamplerV2 as Sampler, Batch
from qiskit.circuit.random import random_circuit

max_circuits = 100
circuits = [pm.run(random_circuit(5, 5)) for _ in range(5 * max_circuits)]
for circuit in circuits:
    circuit.measure_active()
all_partitioned_circuits = []
for i in range(0, len(circuits), max_circuits):
    all_partitioned_circuits.append(circuits[i : i + max_circuits])
jobs = []
start_idx = 0

with Batch(backend=backend):
    sampler = Sampler()
    for partitioned_circuits in all_partitioned_circuits:
        job = sampler.run(partitioned_circuits)
        jobs.append(job)

<Admonition type="caution">
    If you set `backend=backend` in a primitive, the program is run in job mode, even if it's inside a batch or session context. Setting `backend=backend` is deprecated as of Qiskit Runtime v0.24.0.  Instead, use the `mode` parameter.
</Admonition>

## Next steps

<Admonition type="tip" title="Recommendations">
  - Try an example in the [Combine error mitigation options with the estimator primitive](/docs/tutorials/combine-error-mitigation-techniques) tutorial.
  - Review the [Batch API](/docs/api/qiskit-ibm-runtime/batch#batch) reference.
  - Understand the [Job limits](/docs/guides/job-limits) when sending a job to an IBM QPU.
  - Review [execution modes FAQs.](/docs/guides/execution-modes-faq)
</Admonition>